# Historical mineral prices

Run the cells in order. The notebook keeps monthly benchmarks and annual USGS prices in separate CSVs. Months without a monthly source retain a blank price.

In [ ]:
import csv
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path
from xml.etree import ElementTree as ET
from zipfile import ZipFile

## GitHub repository inputs

Run from the cloned repository root or a notebook folder inside it. The notebook locates `Data_econ` from the kernel's working directory. If the clone is elsewhere, set `REPO_ROOT` to its absolute path in the cell below.

The monthly series comes from `Data_econ/CMO/CMO-Historical-Data-Monthly.xlsx`, using the **Monthly Prices** worksheet. The annual reference comes from `Data_econ/MCS2026_Commodities_Data.csv`. Exports go to `rebuilt_prices` under the repository root.

In [ ]:
# Set to Path("/path/to/Semiconductors-DSS") if the clone is elsewhere.
REPO_ROOT = None
if REPO_ROOT is None:
    working_dir = Path.cwd().resolve()
    REPO_ROOT = next(
        (folder for folder in (working_dir, *working_dir.parents) if (folder / "Data_econ").is_dir()),
        None,
    )
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook inside your Semiconductors-DSS clone or set REPO_ROOT above.")

DATA_ECON = REPO_ROOT / "Data_econ"
WORLD_BANK_FILE = DATA_ECON / "CMO" / "CMO-Historical-Data-Monthly.xlsx"
USGS_FILE = DATA_ECON / "MCS2026_Commodities_Data.csv"
OUTPUT_DIR = REPO_ROOT / "rebuilt_prices"

# Edit this list to change the minerals included in the three outputs.
MINERALS = ("Cobalt", "Gallium", "Germanium", "Lithium", "Silicon", "Tantalum", "Tin", "Tungsten")
# GitHub links identify the source files in the repository.
GITHUB_DATA_URL = (
    "https://github.com/EshikaP1/Semiconductors-DSS/blob/"
    "c5670bfcd662aca3d9f330a8ddab3fc10f60c9e8/Data_econ"
)
WORLD_BANK_URL = f"{GITHUB_DATA_URL}/CMO/CMO-Historical-Data-Monthly.xlsx"
USGS_URL = f"{GITHUB_DATA_URL}/MCS2026_Commodities_Data.csv"
NS = {"x": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
REL_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
USD_PER_KG = {
    "dollars per kilogram": Decimal("1"),
    "dollars per metric ton": Decimal("0.001"),
    "dollars per pound": Decimal("1") / Decimal("0.45359237"),
    "cents per pound": Decimal("0.01") / Decimal("0.45359237"),
}
MONTHLY_MISSING = {"", "…", "...", "..", "NA", "N/A", "—"}

## Read monthly prices and check coverage

In [ ]:
def monthly_sheet(path):
    """Read the Monthly Prices worksheet using only the Python standard library."""
    with ZipFile(path) as book:
        workbook = ET.fromstring(book.read("xl/workbook.xml"))
        sheet = workbook.find(".//x:sheet[@name='Monthly Prices']", NS)
        if sheet is None:
            raise ValueError("World Bank workbook has no Monthly Prices worksheet")
        sheet_id = sheet.attrib[f"{{{REL_NS}}}id"]
        relationships = ET.fromstring(book.read("xl/_rels/workbook.xml.rels"))
        target = next(rel.attrib["Target"] for rel in relationships if rel.attrib["Id"] == sheet_id)
        sheet_path = target.lstrip("/") if target.startswith("/") else f"xl/{target}"
        data = ET.fromstring(book.read(sheet_path))
        strings = []
        if "xl/sharedStrings.xml" in book.namelist():
            shared = ET.fromstring(book.read("xl/sharedStrings.xml"))
            strings = ["".join(t.text or "" for t in item.findall(".//x:t", NS)) for item in shared]

    rows = {}
    for row in data.findall(".//x:row", NS):
        cells = {}
        for cell in row.findall("x:c", NS):
            column = cell.attrib["r"].rstrip("0123456789")
            value = cell.findtext("x:v", "", NS)
            if cell.get("t") == "s":
                value = strings[int(value)]
            elif cell.get("t") == "inlineStr":
                value = "".join(t.text or "" for t in cell.findall(".//x:t", NS))
            cells[column] = value.strip()
        rows[int(row.attrib["r"])] = cells
    return rows


def read_monthly_prices(path):
    rows = monthly_sheet(path)
    columns = {label: column for column, label in rows[5].items() if label in MINERALS}
    for mineral, column in columns.items():
        if rows[6].get(column) != "($/mt)":
            raise ValueError(f"Unexpected World Bank unit for {mineral}")

    source_rows = {}
    for row in rows.values():
        period = row.get("A", "")
        if re.fullmatch(r"\d{4}M(0[1-9]|1[0-2])", period):
            year, month = map(int, period.split("M"))
            index = year * 12 + month - 1
            if index in source_rows:
                raise ValueError(f"Duplicate World Bank month: {period}")
            source_rows[index] = row
    if not source_rows:
        raise ValueError("No World Bank months found")

    monthly = []
    for index in range(min(source_rows), max(source_rows) + 1):
        year, month = divmod(index, 12)
        date = f"{year:04d}-{month + 1:02d}-01"
        row = source_rows.get(index, {})
        for mineral in MINERALS:
            if mineral not in columns:
                monthly.append((date, mineral, None, "", "no_monthly_series", ""))
                continue
            raw = row.get(columns[mineral], "")
            price = None if raw.upper() in MONTHLY_MISSING else Decimal(raw) / 1000
            if price is not None and (not price.is_finite() or price <= 0):
                raise ValueError(f"Invalid monthly price for {mineral} in {date}: {raw}")
            status = "observed" if price is not None else "missing_in_source"
            monthly.append((date, mineral, price, "benchmark_price", status, WORLD_BANK_URL))
    return monthly, columns


def coverage_rows(monthly, columns):
    for mineral in MINERALS:
        rows = [row for row in monthly if row[1] == mineral]
        observed = [row[0] for row in rows if row[2] is not None]
        first = observed[0] if observed else ""
        last = observed[-1] if observed else ""
        gaps = sum(row[2] is None and first <= row[0] <= last for row in rows) if observed else ""
        note = "World Bank monthly benchmark" if mineral in columns else "USGS annual only in assigned sources"
        yield mineral, rows[0][0], rows[-1][0], len(observed), len(rows) - len(observed), first, last, gaps, note

## Read annual USGS prices

In [ ]:
def read_usgs_prices(path):
    annual = []
    with path.open(encoding="cp1252", newline="") as file:
        for row in csv.DictReader(file):
            mineral = row["MCS chapter"].title()
            if mineral not in MINERALS or row["Statistics"] != "Price":
                continue
            original = row["Value"]
            unit = row["Unit"]
            try:
                price = Decimal(original.replace(",", ""))
            except InvalidOperation:
                price = None
            # Keep unconverted series, such as tungsten, in their source units.
            factor = USD_PER_KG.get(unit)
            normalized = round(price * factor, 8) if price is not None and factor is not None else None
            annual.append((
                row["Year"], mineral, row["Country"], normalized,
                row["Statistics_detail"], original, unit, row["Notes"], USGS_URL,
            ))
    return sorted(annual, key=lambda row: (row[1], row[4], row[0]))


def write_csv(path, header, rows):
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(rows)

## Build and preview the data

The coverage preview prints the number of observed and missing months for each mineral.

In [ ]:
monthly, columns = read_monthly_prices(WORLD_BANK_FILE)
annual = read_usgs_prices(USGS_FILE)

print(f"{len(monthly):,} monthly rows and {len(annual):,} annual price rows")
for row in coverage_rows(monthly, columns):
    print(f"{row[0]}: {row[3]:,} observed, {row[4]:,} missing")

## Export the three CSVs

Rerunning this cell replaces the three CSVs in the configured output directory.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_csv(
    OUTPUT_DIR / "monthly_mineral_prices.csv",
    ("month", "mineral", "price_usd_per_kg", "price_type", "status", "source_url"),
    monthly,
)
write_csv(
    OUTPUT_DIR / "monthly_price_coverage.csv",
    ("mineral", "table_start", "table_end", "observed_months", "missing_months_in_table",
     "first_observation", "last_observation", "missing_between_first_and_last", "coverage_note"),
    coverage_rows(monthly, columns),
)
write_csv(
    OUTPUT_DIR / "usgs_annual_price_reference.csv",
    ("year", "mineral", "country", "price_usd_per_kg", "price_series",
     "original_value", "original_unit", "source_notes", "source_url"),
    annual,
)


print(f"Saved three CSVs to {OUTPUT_DIR.resolve()}")